# **Dashboard**

Este dashboard é uma aplicação interativa que permite analisar os dados das vendas online, presentes no dataset "Online_retail_listing" de uma forma mais visual e detalhada.
Vou apresentar aqui, um guia de como funciona o Dashboard:

1. Título: "Online Retail Dashboard"

2. Na parte superior do dashboard, são apresentados quatro blocos com métricas-chave:
- Total Sales: Mostra o total de vendas no período selecionado.
- Total Transactions: Exibe o número total de transações realizadas.
- Avg Order Value: Calcula o valor médio por pedido.
- Unique Customers: Indica o número de clientes únicos.
- Estes números são atualizados automaticamente com base nos filtros aplicados (países e intervalo de datas).

3. Filtros: à esquerda, na seção de filtros, pode-se:
- Selecionar País(es):Utilizando o dropdown para escolher um ou mais países, o que filtra os dados para mostrar apenas as vendas relacionadas aos países selecionados.
- Selecionar Intervalo de Datas: Utilizando o seletor de datas para definir um período de tempo específico. (Os gráficos e as métricas serão atualizados de acordo com as datas selecionadas.)

4. Gráfico de Tendências (Sales Trends): Exibe as vendas mensais num formato de gráfico de linhas, o qual muda dinamicamente de acordo com os filtros.

5. Produtos e Clientes: 
- Top Products: Um gráfico de barras que lista os 10 produtos mais vendidos (em termos de receita). Ideal para identificar quais itens são mais populares.
- Top Customers: Outro gráfico de barras que mostra os 10 clientes que mais gastaram em valor. Útil para entender quais os clientes mais valiosos.
(Ambas estas informações também são alteradas dinâmicamente de acordo com os filtros selecionados.)
(Acabei por só selecionar o Top 10 de cada opção de modo a tornar o dashboard vizualemente mais "claro" de modo a evitar a sobrecarga vizual.)

6. Bubble Chart (Gráfico de Bolhas): Exibe os 10 produtos mais frequentes. Cada bolha representa um produto, tendo cada uma o tamanho indicando com base na sua frequência relativa.
- Passando o cursor sobre as bolhas, estas revelam mais detalhes sobre os artigos.
- Clicando numa bolha é exibido o nome do completo do artigo, pois no gráfico só aparece uma abreviação do nome
Descrição Completa:

7. Aplicações Práticas deste dashboard
*Análise Regional:* Selecionando países específicos para entender o desempenho por região.
*Monitorização de Tendências:* Utilizando o gráfico de tendências para observar picos ou quedas sazonais.
*Identificação de Produtos-Chave:* Analisando o gráfico de produtos para ajustar o stock ou a estratégia de marketing.
*Segmentação de Clientes:* Examinando os clientes mais valiosos.

In [2]:
import dash
from dash import dcc, html, Input, Output
import dash_bootstrap_components as dbc
import pandas as pd
import plotly.express as px
import collections

# Load the data
file_path = "online_retail_listing.csv"
data = pd.read_csv(file_path, delimiter=';', decimal=',', encoding='latin1')

# Data Cleaning and Transformation
data.columns = data.columns.str.strip()
data = data.dropna(subset=['Description', 'Customer ID'])
data['Quantity'] = data['Quantity'].clip(lower=0)
data['Price'] = data['Price'].clip(lower=0)
data['TotalValue'] = data['Quantity'] * data['Price']
data['Country'] = data['Country'].str.upper()
data['InvoiceDate'] = pd.to_datetime(data['InvoiceDate'], format='%d.%m.%Y %H:%M')

# App initialization
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Layout
app.layout = html.Div([
    dbc.Container([
        html.H1("Online Retail Dashboard", style={'textAlign': 'center'}),

        dbc.Row([
            dbc.Col([html.H3("Key Metrics")], width=12)
        ]),

        dbc.Row([
            dbc.Col([html.Div(id='total-sales', className='metric-card')], width=3),
            dbc.Col([html.Div(id='total-transactions', className='metric-card')], width=3),
            dbc.Col([html.Div(id='avg-order-value', className='metric-card')], width=3),
            dbc.Col([html.Div(id='unique-customers', className='metric-card')], width=3),
        ]),

        dbc.Row([
            dbc.Col([
                html.H3("Filters"),
                dcc.Dropdown(
                    id='country-selector',
                    options=[{'label': country, 'value': country} for country in data['Country'].unique()],
                    value=None,
                    placeholder="Select a Country",
                    multi=True,
                ),
                dcc.DatePickerRange(
                    id='date-picker',
                    start_date=data['InvoiceDate'].min(),
                    end_date=data['InvoiceDate'].max(),
                    display_format='YYYY-MM-DD'
                )
            ], width=4),

            dbc.Col([
                html.H3("Sales Trends"),
                dcc.Graph(id='sales-trends')
            ], width=8),
        ]),

        dbc.Row([
            dbc.Col([
                html.H3("Top Products"),
                dcc.Graph(id='top-products')
            ], width=6),
            dbc.Col([
                html.H3("Top Customers"),
                dcc.Graph(id='top-customers')
            ], width=6),
        ]),

        dbc.Row([
            dbc.Col([
                html.H3("Bubble Chart: Top 10 Product Phrases"),
                dcc.Graph(id='word-cloud')
            ], width=8),
            dbc.Col([
                html.H3("Full Description"),
                html.Div(
                    id='description-display',
                    style={
                        'padding': '10px',
                        'border': '1px solid #ccc',
                        'border-radius': '5px',
                        'background-color': '#f9f9f9'
                    }
                )
            ], width=4),
        ]),
    ])
])

# Callbacks
@app.callback(
    [Output('total-sales', 'children'),
     Output('total-transactions', 'children'),
     Output('avg-order-value', 'children'),
     Output('unique-customers', 'children')],
    [Input('country-selector', 'value'),
     Input('date-picker', 'start_date'),
     Input('date-picker', 'end_date')]
)
def update_metrics(selected_countries, start_date, end_date):
    filtered_data = data[(data['InvoiceDate'] >= start_date) & (data['InvoiceDate'] <= end_date)]
    if selected_countries:
        filtered_data = filtered_data[filtered_data['Country'].isin(selected_countries)]

    total_sales = filtered_data['TotalValue'].sum()
    total_transactions = filtered_data['Invoice'].nunique()
    avg_order_value = total_sales / total_transactions if total_transactions else 0
    unique_customers = filtered_data['Customer ID'].nunique()

    return (
        f"Total Sales: {total_sales:,.2f}£ (libras)",
        f"Total Transactions: {total_transactions}",
        f"Avg Order Value: {avg_order_value:,.2f}£ (libras)",
        f"Unique Customers: {unique_customers}"
    )

@app.callback(
    Output('sales-trends', 'figure'),
    [Input('country-selector', 'value'),
     Input('date-picker', 'start_date'),
     Input('date-picker', 'end_date')]
)
def update_sales_trends(selected_countries, start_date, end_date):
    filtered_data = data[(data['InvoiceDate'] >= start_date) & (data['InvoiceDate'] <= end_date)]
    if selected_countries:
        filtered_data = filtered_data[filtered_data['Country'].isin(selected_countries)]

    trends = filtered_data.groupby(filtered_data['InvoiceDate'].dt.to_period('M'))['TotalValue'].sum()
    fig = px.line(trends, x=trends.index.astype(str), y=trends.values,
                  title="Monthly Sales Trends", labels={'x': 'Month', 'y': 'Total Sales'})
    return fig

@app.callback(
    Output('top-products', 'figure'),
    [Input('country-selector', 'value'),
     Input('date-picker', 'start_date'),
     Input('date-picker', 'end_date')]
)
def update_top_products(selected_countries, start_date, end_date):
    filtered_data = data[(data['InvoiceDate'] >= start_date) & (data['InvoiceDate'] <= end_date)]
    if selected_countries:
        filtered_data = filtered_data[filtered_data['Country'].isin(selected_countries)]

    top_products = filtered_data.groupby('Description')['TotalValue'].sum().nlargest(10)
    fig = px.bar(top_products, x=top_products.index, y=top_products.values,
                 title="Top 10 Products by Sales", labels={'x': 'Product', 'y': 'Total Sales'})
    return fig

@app.callback(
    Output('top-customers', 'figure'),
    [Input('country-selector', 'value'),
     Input('date-picker', 'start_date'),
     Input('date-picker', 'end_date')]
)
def update_top_customers(selected_countries, start_date, end_date):
    filtered_data = data[(data['InvoiceDate'] >= start_date) & (data['InvoiceDate'] <= end_date)]
    if selected_countries:
        filtered_data = filtered_data[filtered_data['Country'].isin(selected_countries)]

    top_customers = filtered_data.groupby('Customer ID')['TotalValue'].sum().nlargest(10)
    fig = px.bar(top_customers, x=top_customers.index.astype(str), y=top_customers.values,
                 title="Top 10 Customers by Sales", labels={'x': 'Customer ID', 'y': 'Total Sales'})
    return fig

@app.callback(
    [Output('word-cloud', 'figure'),
     Output('description-display', 'children')],
    [Input('country-selector', 'value'),
     Input('date-picker', 'start_date'),
     Input('date-picker', 'end_date'),
     Input('word-cloud', 'clickData')]
)
def update_bubble_chart(selected_countries, start_date, end_date, click_data):
    filtered_data = data[(data['InvoiceDate'] >= start_date) & (data['InvoiceDate'] <= end_date)]
    if selected_countries:
        filtered_data = filtered_data[filtered_data['Country'].isin(selected_countries)]

    # Extract the first three words from each description
    filtered_data['FirstThreeWords'] = filtered_data['Description'].str.split().str[:3].str.join(' ')
    
    # Count the frequency of these three-word phrases
    phrase_counts = filtered_data['FirstThreeWords'].value_counts().reset_index()
    phrase_counts.columns = ['FirstThreeWords', 'Frequency']
    phrase_counts['Size'] = phrase_counts['Frequency'] ** 0.5

    # Limit to top 10 phrases
    top_10_phrases = phrase_counts.head(10)

    # Create the bubble chart
    fig = px.scatter(
        top_10_phrases, 
        x='FirstThreeWords', 
        y='Frequency', 
        size='Size', 
        color='FirstThreeWords',
        text='FirstThreeWords',
        title="Bubble Chart: Top 10 Product Phrases",
        labels={'Frequency': 'Phrase Frequency', 'FirstThreeWords': 'Product Phrases'}
    )
    fig.update_traces(textposition='top center')
    fig.update_layout(xaxis=dict(title="Product Phrases", tickangle=-45))

    # Display the full description of the clicked article
    if click_data:
        selected_phrase = click_data['points'][0]['x']
        descriptions = filtered_data[filtered_data['FirstThreeWords'] == selected_phrase]['Description'].unique()
        if len(descriptions) > 0:
            description_text = descriptions[0]  # Show only the first description
        else:
            description_text = "No description available for this phrase."
    else:
        description_text = "Click on a bubble to view an article's name."

    return fig, description_text



# Run app
if __name__ == "__main__":
    app.run_server(debug=True)


Para o desenvolvimento deste Dashboard foi utilizado o chatGPT para auxilio e tratamento de erros, e para o seu design teve-se por base as informações dos seguintes websites:
- https://www.tableau.com/dashboard/sales-dashboard-examples-and-templates;
- https://www.geckoboard.com/dashboard-examples/sales/;
